In [1]:
import pandas as pd

web_data = pd.read_csv("cleaned_web_data.csv")
web_data[["client_id", "visit_id", "process_step", "date_time", "variation"]].head(10)

,client_id,visit_id,process_step,date_time,variation
0,555,637149525_38041617439_716659,confirm,2017-04-15 13:00:34,Test
1,555,637149525_38041617439_716659,step_3,2017-04-15 13:00:14,Test
2,555,637149525_38041617439_716659,step_2,2017-04-15 12:58:35,Test
3,555,637149525_38041617439_716659,step_1,2017-04-15 12:58:03,Test
4,555,637149525_38041617439_716659,start,2017-04-15 12:57:56,Test
5,647,40369564_40101682850_311847,confirm,2017-04-12 15:47:45,Test
6,647,40369564_40101682850_311847,step_3,2017-04-12 15:45:02,Test
7,647,40369564_40101682850_311847,step_2,2017-04-12 15:41:53,Test
8,647,40369564_40101682850_311847,step_1,2017-04-12 15:41:35,Test
9,647,40369564_40101682850_311847,start,2017-04-12 15:41:28,Test


In [2]:
# We check the data type of date_time & correct it
web_data["date_time"].dtype

dtype('O')

In [3]:
web_data["date_time"] = pd.to_datetime(web_data["date_time"])
web_data["date_time"].dtype

dtype('<M8[ns]')

In [4]:
# We calculate the duration of every step
# First, we group the records by client and visit with the earliest timestamp first

timed_steps = (
    web_data
    .sort_values(["client_id", "visit_id", "date_time"])
    .reset_index(drop=True)
)

timed_steps[["client_id", "visit_id", "process_step", "date_time", "variation"]].head(10)

,client_id,visit_id,process_step,date_time,variation
0,555,637149525_38041617439_716659,start,2017-04-15 12:57:56,Test
1,555,637149525_38041617439_716659,step_1,2017-04-15 12:58:03,Test
2,555,637149525_38041617439_716659,step_2,2017-04-15 12:58:35,Test
3,555,637149525_38041617439_716659,step_3,2017-04-15 13:00:14,Test
4,555,637149525_38041617439_716659,confirm,2017-04-15 13:00:34,Test
5,647,40369564_40101682850_311847,start,2017-04-12 15:41:28,Test
6,647,40369564_40101682850_311847,step_1,2017-04-12 15:41:35,Test
7,647,40369564_40101682850_311847,step_2,2017-04-12 15:41:53,Test
8,647,40369564_40101682850_311847,step_3,2017-04-12 15:45:02,Test
9,647,40369564_40101682850_311847,confirm,2017-04-12 15:47:45,Test


In [5]:
# We add a new column with the following timestamp so we can compare them after

timed_steps["next_step_time"] = (
    timed_steps.groupby(["client_id", "visit_id"])["date_time"].shift(-1)
)

timed_steps[["process_step", "date_time", "next_step_time", "variation"]].head(10)

,process_step,date_time,next_step_time,variation
0,start,2017-04-15 12:57:56,2017-04-15 12:58:03,Test
1,step_1,2017-04-15 12:58:03,2017-04-15 12:58:35,Test
2,step_2,2017-04-15 12:58:35,2017-04-15 13:00:14,Test
3,step_3,2017-04-15 13:00:14,2017-04-15 13:00:34,Test
4,confirm,2017-04-15 13:00:34,NaT,Test
5,start,2017-04-12 15:41:28,2017-04-12 15:41:35,Test
6,step_1,2017-04-12 15:41:35,2017-04-12 15:41:53,Test
7,step_2,2017-04-12 15:41:53,2017-04-12 15:45:02,Test
8,step_3,2017-04-12 15:45:02,2017-04-12 15:47:45,Test
9,confirm,2017-04-12 15:47:45,NaT,Test


In [8]:
# We calculate the time passed between steps and add a new column to display it

timed_steps["step_duration"] = (
    timed_steps["next_step_time"] - timed_steps["date_time"]
).dt.total_seconds()

timed_steps[["process_step", "date_time", "next_step_time", "step_duration", "variation"]].head(10)

,process_step,date_time,next_step_time,step_duration,variation
0,start,2017-04-15 12:57:56,2017-04-15 12:58:03,7.0,Test
1,step_1,2017-04-15 12:58:03,2017-04-15 12:58:35,32.0,Test
2,step_2,2017-04-15 12:58:35,2017-04-15 13:00:14,99.0,Test
3,step_3,2017-04-15 13:00:14,2017-04-15 13:00:34,20.0,Test
4,confirm,2017-04-15 13:00:34,NaT,NaN,Test
5,start,2017-04-12 15:41:28,2017-04-12 15:41:35,7.0,Test
6,step_1,2017-04-12 15:41:35,2017-04-12 15:41:53,18.0,Test
7,step_2,2017-04-12 15:41:53,2017-04-12 15:45:02,189.0,Test
8,step_3,2017-04-12 15:45:02,2017-04-12 15:47:45,163.0,Test
9,confirm,2017-04-12 15:47:45,NaT,NaN,Test


In [ ]:
# We want to check how many sessions reached a specific step in the user journey

NameError: name 'step_counts' is not defined